# 🗓️ 31일차 스터디 노트북 — 원형 이중 연결 리스트 (08장 완결편)

**오늘 범위**: 08-4 원형 이중 연결 리스트 — 원형 리스트 · 이중 연결 리스트 · **더미 노드** · 실습 8-5 `DoubleLinkedList` · 양방향 이터레이터 · 보충수업 8-4 파이썬의 대입

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[그림]** 화살표 직접 그리기 · **[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]**

---

## 🎯 오늘은 "약점 청산의 날"

29~30일차에서 연결 리스트의 약점 세 개를 발견했지. 오늘 **세 개를 한 번에** 없애.

| 약점 | 언제 발견 | 오늘의 해법 |
|---|---|---|
| ① `add_last` 가 O(n) (꼬리를 매번 찾음) | 29일차 21번 | **원형** → `head.prev` 가 곧 꼬리 |
| ② 앞쪽 노드로 못 감 (`pre` 커서 필요) | 29일차 3·10번 | **이중** → `prev` 포인터 |
| ③ "빈 리스트일 때" 분기가 계속 나옴 | 29일차 7·8번 | **더미 노드** → 분기 자체가 사라짐 |

교재 356p:
> **"원형 리스트가 연결 리스트와 가장 크게 다른 점은 꼬리 노드(F)의 뒤쪽 포인터가 None이 아니라 머리 노드의 포인터값이 된다는 것입니다."**
> **"연결 리스트의 가장 큰 단점은 뒤쪽 노드를 찾기 쉬운 반면 앞쪽 노드를 찾기 어렵다는 것입니다. 이 단점을 개선한 리스트 구조가 이중 연결 리스트입니다."**

## 오늘의 하이라이트 두 개

> **🔥 ① 더미 노드가 모든 분기를 없앤다** (7~9번)
> 29일차 코드에는 `if self.head is None:` 이 계속 나왔는데, 오늘은 **한 번도 안 나와.**
>
> **🔥 ② 파이썬의 연속 대입 함정** (17번, 보충수업 8-4)
> `a = b = node` 를 한 줄로 쓰면 **리스트가 통째로 깨져.** 다른 언어에서는 되는데 파이썬에서는 안 돼.

## 진행 순서
**세 개념(1~4) → 더미 노드(5~9) → 삽입·삭제 그림(10~13) → 코드 구현(14~18) → 성능과 총정리(19~21)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from __future__ import annotations
from typing import Any
import time

class Node:
    """원형 이중 연결 리스트용 노드 클래스"""
    def __init__(self, data: Any = None, prev: Node = None, next: Node = None) -> None:
        self.data = data
        self.prev = prev or self      # 앞쪽 포인터 (🔥 or 연산자! 3번)
        self.next = next or self      # 뒤쪽 포인터


class DoubleLinkedList:
    """원형 이중 연결 리스트 클래스"""
    def __init__(self) -> None:
        self.head = self.current = Node()   # 더미 노드를 생성
        self.no = 0

    def __len__(self) -> int:
        return self.no

    def is_empty(self) -> bool:
        return self.head.next is self.head

    def search(self, data: Any) -> Any:
        cnt = 0
        ptr = self.head.next                # 🔥 head가 아니라 head.next!
        while ptr is not self.head:
            if data == ptr.data:
                self.current = ptr
                return cnt
            cnt += 1
            ptr = ptr.next
        return -1

    def __contains__(self, data: Any) -> bool:
        return self.search(data) >= 0

    def print_current_node(self) -> None:
        if self.is_empty():
            print('주목 노드는 없습니다.')
        else:
            print(self.current.data)

    def print(self) -> None:
        ptr = self.head.next
        while ptr is not self.head:
            print(ptr.data); ptr = ptr.next

    def print_reverse(self) -> None:
        ptr = self.head.prev
        while ptr is not self.head:
            print(ptr.data); ptr = ptr.prev

    def next(self) -> bool:
        if self.is_empty() or self.current.next is self.head:
            return False
        self.current = self.current.next
        return True

    def prev(self) -> bool:
        if self.is_empty() or self.current.prev is self.head:
            return False
        self.current = self.current.prev
        return True

    def add(self, data: Any) -> None:
        """주목 노드 바로 뒤에 노드를 삽입"""
        node = Node(data, self.current, self.current.next)
        self.current.next.prev = node       # 🔥 순서 주의! (17번)
        self.current.next = node
        self.current = node
        self.no += 1

    def add_first(self, data: Any) -> None:
        self.current = self.head            # 더미 바로 뒤에 삽입
        self.add(data)

    def add_last(self, data: Any) -> None:
        self.current = self.head.prev       # 🔥 꼬리를 O(1)에 찾는다!
        self.add(data)

    def remove_current_node(self) -> None:
        if not self.is_empty():
            self.current.prev.next = self.current.next
            self.current.next.prev = self.current.prev
            self.current = self.current.prev
            self.no -= 1
            if self.current is self.head:   # 더미면 머리로 보정
                self.current = self.head.next

    def remove(self, p: Node) -> None:
        ptr = self.head.next
        while ptr is not self.head:
            if ptr is p:
                self.current = p
                self.remove_current_node()
                break
            ptr = ptr.next

    def remove_first(self) -> None:
        self.current = self.head.next
        self.remove_current_node()

    def remove_last(self) -> None:
        self.current = self.head.prev
        self.remove_current_node()

    def clear(self) -> None:
        while not self.is_empty():
            self.remove_first()
        self.no = 0

    def __iter__(self):
        return DoubleLinkedListIterator(self.head)

    def __reversed__(self):
        return DoubleLinkedListReverseIterator(self.head)


class DoubleLinkedListIterator:
    def __init__(self, head: Node):
        self.head = head
        self.current = head.next
    def __iter__(self): return self
    def __next__(self) -> Any:
        if self.current is self.head:
            raise StopIteration
        data = self.current.data
        self.current = self.current.next
        return data


class DoubleLinkedListReverseIterator:
    def __init__(self, head: Node):
        self.head = head
        self.current = head.prev
    def __iter__(self): return self
    def __next__(self) -> Any:
        if self.current is self.head:
            raise StopIteration
        data = self.current.data
        self.current = self.current.prev
        return data


# ===== 실험용 시각화 =====
def draw(l, title="", limit=12):
    """정방향·역방향을 모두 그린다 (원형이 깨졌는지도 감지)"""
    if title: print(title)
    f, p, c = [], l.head.next, 0
    while p is not l.head and c < limit:
        f.append(f"[{p.data}]{'*' if p is l.current else ''}"); p = p.next; c += 1
    broke_f = c >= limit
    b, p, c = [], l.head.prev, 0
    while p is not l.head and c < limit:
        b.append(f"[{p.data}]"); p = p.prev; c += 1
    broke_b = c >= limit
    cur = '더미' if l.current is l.head else (l.current.data if l.current else None)
    print(f"  정방향: 더미 ⇄ " + " ⇄ ".join(f) + (" ⇄ ... 💥끊김" if broke_f else " ⇄ 더미"))
    print(f"  역방향: 더미 ⇄ " + " ⇄ ".join(b) + (" ⇄ ... 💥끊김" if broke_b else " ⇄ 더미"))
    print(f"  no={l.no}  current={cur}  is_empty={l.is_empty()}\n")

def make(*vals):
    l = DoubleLinkedList()
    for v in vals: l.add_last(v)
    return l

print("준비 완료 ✅\n")
draw(make('A','B','C','D','E'), "예시: A~E")

---
# 🔁 [Remind] 워밍업 — 29~30일차 되감기

오늘 코드가 왜 이렇게 짧아졌는지 이해하려면 **어제까지의 불편함**을 정확히 기억해야 해.

### R-1. 🟢 [설명] 29일차의 불편함 세 가지

**(가) `add_last` (29일차 8번)**
```python
def add_last(self, data):
    if self.head is None:          # ← 분기 1
        self.add_first(data)
    else:
        ptr = self.head
        while ptr.next is not None:    # ← 꼬리 찾기 O(n)
            ptr = ptr.next
        ptr.next = self.current = Node(data, None)
```
- `while` 이 하는 일은? ①____ 시간 복잡도는? ②____

**(나) `remove_last` (29일차 10번)**
```python
ptr = self.head
pre = self.head              # ← 커서가 왜 둘?
while ptr.next is not None:
    pre = ptr
    ptr = ptr.next
```
- `pre` 가 필요한 이유는? ③________________

**(다) 빈 리스트 분기**
- 29일차 코드에서 `if self.head is None:` 또는 `if self.head is not None:` 이 **몇 개 함수**에 나왔지? 세어봐. ④____

### R-2. 🟡 [설명] 30일차 보충수업 8-3 소환

30일차 19번에서 배운 파이썬 논리 연산자:
> `x or y` — x를 평가하여 **참**이면 그 값을 생성. 그렇지 않으면 y를 평가하여 그 값을 생성

오늘 `Node.__init__` 에 이게 나와:
```python
self.prev = prev or self
self.next = next or self
```

- `prev` 가 `None` 일 때 `self.prev` 에 들어가는 건? ①____
- `prev` 가 실제 노드일 때는? ②____
- 이걸 `if` 문으로 풀어 쓰면? ③________________

*(답을 적은 뒤 실행)*

In [ ]:
print("[R-2] prev or self 동작 확인")
a = Node('A')
print(f"  Node('A') → prev is self? {a.prev is a}, next is self? {a.next is a}")
print("  → 인자를 안 주면 자기 자신을 가리킨다 (원형의 씨앗!)\n")

b = Node('B', a, a)
print(f"  Node('B', a, a) → prev.data={b.prev.data}, next.data={b.next.data}")
print("  → 인자를 주면 그걸 그대로 쓴다\n")

print("  if문으로 풀어 쓰면:")
print("    self.prev = prev if prev is not None else self")
print("  💡 30일차 19번의 or 연산자가 여기서 실전 사용됨")

---
# 🎯 PART 1 — 세 가지 개념 (1~4번)

### 1. 🟢 [그림] 원형 리스트

교재 356p [그림 8-20].

**일반 연결 리스트 (29일차)**
```
head → [A] → [B] → [C] → None
```

**원형 리스트**
```
head → [A] → [B] → [C] ─┐
        ↑                │
        └────────────────┘
```

- 꼬리 노드 `C` 의 `next` 는? ①____
- 그럼 **"리스트의 끝"을 어떻게 판단**하지? 29일차는 `ptr is None` 이었는데, 원형에서는? ②________________
- 🔥 원형이면 **`head` 하나로 꼬리에 접근**할 수 있어. 어떻게? ③________________
  → 이게 `add_last` 를 O(1)로 만드는 열쇠야.
- 교재 356p: **"고리 모양으로 늘어선 데이터를 표현하는 데 알맞은 리스트 구조입니다."**
  → 어떤 데이터가 "고리 모양"일까? 예를 두 개 들어봐.

### 2. 🟢 [그림] 이중 연결 리스트

교재 356p [그림 8-21], 357p [그림 8-22].

```
        ┌──next──┐   ┌──next──┐
head → [A]      [B]      [C]
        └──prev──┘   └──prev──┘
```

- `Node` 의 필드가 **3개**로 늘었어: ①____, ②____, ③____
- 29일차 3번에서 **"연결 리스트의 결정적 약점"** 이 뭐였지? ④________________
- `prev` 가 생기면 R-1 (나)의 `pre` 커서가 필요 없어져. 왜? ⑤________________
- 교재 356p: **"이중 연결 리스트는 양방향 리스트(bidirectional linked list)라고도 합니다."**

### 3. 🟡 [설명] 원형 + 이중 = 오늘의 구조

교재 357p [그림 8-23]. 둘을 합치면:

```
   ┌─────────── next ───────────┐
   ↓                            │
 [더미] ⇄ [A] ⇄ [B] ⇄ [C] ⇄ ────┘
   ↑                            │
   └─────────── prev ───────────┘
```

- 꼬리 `C` 의 `next` 는 ①____, 더미의 `prev` 는 ②____
- 그럼 `head.prev` 는 항상 **무엇**을 가리켜? ③____
- `head.next` 는? ④____
- 🔥 **머리와 꼬리를 둘 다 O(1)에 얻을 수 있어.** 29일차에서 꼬리를 찾는 데 O(n)이 걸렸던 걸 생각하면 큰 차이지.

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C','D','E')
print("원형 확인 — 한 바퀴 돌면 제자리로 오는가?")
p = l.head; path = []
for _ in range(7):
    path.append('더미' if p is l.head else p.data)
    p = p.next
print("  next만 따라가기:", ' → '.join(path))
print("  → 7번 갔더니 더미가 다시 나온다 = 원형 ✅\n")

p = l.head; path = []
for _ in range(7):
    path.append('더미' if p is l.head else p.data)
    p = p.prev
print("  prev만 따라가기:", ' → '.join(path))
print("  → 역방향으로도 한 바퀴 = 이중 ✅\n")

print(f"머리와 꼬리를 O(1)에:")
print(f"  head.next.data = {l.head.next.data}   ← 머리")
print(f"  head.prev.data = {l.head.prev.data}   ← 꼬리")
print("\n29일차라면 꼬리를 찾는 데 while로 n칸을 걸어야 했다 (O(n))")

### 4. 🟡 [손] 교재 360p 참조식 표 채우기

교재 360p에 이런 표가 있어. **직접 채워봐.**

리스트가 `더미 ⇄ A ⇄ B ⇄ C ⇄ D ⇄ E ⇄ 더미` 이고, 변수 `a, b, c, d, e` 가 각각 노드 A~E를 참조해.

| 노드 | 식 1 | 식 2 | 식 3 | 식 4 |
|---|---|---|---|---|
| 더미 | `head` | `e.next` | ① | ② |
| A | `head.next` | ③ | ④ | `c.prev.prev` |
| B | ⑤ | `a.next` | `head.next.next` | ⑥ |
| C | `b.next` | ⑦ | `d.prev` | ⑧ |
| D | `c.next` | `b.next.next` | ⑨ | ⑩ |
| E | ⑪ | `c.next.next` | `head.prev` | ⑫ |

**힌트**: `next` 를 k번 가면 k칸 뒤, `prev` 를 k번 가면 k칸 앞. 원형이라 **더미를 지나 한 바퀴 돌 수도** 있어.

그리고 교재 362p "조금만 더" — 위치 판별식:
```
p.prev is head        # p는 ⑬________________
p.prev.prev is head   # p는 ⑭________________
p.next is head        # p는 ⑮________________
p.next.next is head   # p는 ⑯________________
```

*(채운 뒤 실행)*

In [ ]:
l = make('A','B','C','D','E')
a = l.head.next; b = a.next; c = b.next; d = c.next; e = d.next
head = l.head

checks = [
    ("더미", head, [("e.next", e.next), ("d.next.next", d.next.next),
                    ("a.prev", a.prev), ("b.prev.prev", b.prev.prev)]),
    ("A", a, [("head.next", head.next), ("e.next.next", e.next.next),
              ("b.prev", b.prev), ("c.prev.prev", c.prev.prev)]),
    ("E", e, [("d.next", d.next), ("c.next.next", c.next.next),
              ("head.prev", head.prev), ("a.prev.prev", a.prev.prev)]),
]
for name, target, exprs in checks:
    marks = "  ".join(f"{s}:{'✅' if v is target else '❌'}" for s, v in exprs)
    print(f"  {name:4s} → {marks}")

print("\n[위치 판별식]")
for nm, node in (("A", a), ("B", b), ("D", d), ("E", e)):
    print(f"  {nm}: 머리? {node.prev is head}  앞2번째? {node.prev.prev is head}  "
          f"꼬리? {node.next is head}  뒤2번째? {node.next.next is head}")

---
# 👻 PART 2 — 더미 노드 (5~9번)

> **오늘의 첫 번째 하이라이트.** 교재는 이걸 담담하게 지나가는데, 사실 **코드를 절반으로 줄이는 아이디어**야.

### 5. 🟢 [설명] 더미 노드란

교재 359p:
> **"`__init__()` 함수는 비어 있는 원형 이중 연결 리스트를 생성합니다. 이때 [그림 8-24]처럼 데이터가 없는 노드를 1개 만듭니다. 이 노드는 삽입과 삭제를 원활하게 처리하기 위해 리스트의 맨 앞에 계속 존재하는 더미 노드입니다."**

```python
def __init__(self) -> None:
    self.head = self.current = Node()    # 더미 노드를 생성
    self.no = 0
```

**빈 리스트의 모습** (교재 [그림 8-24])
```
head → [더미] ─┐
        ↑ ↓    │
        └──────┘   (자기 자신을 가리킴)
```

- 더미 노드의 `data` 는? ①____
- 더미의 `prev` 와 `next` 는 각각 무엇을 가리켜? ②____
  → R-2의 `prev or self` 가 이걸 만들어냈지!
- `no` 가 0인데 **노드는 1개** 있어. 모순 아니야? ③________________
- 🔥 **더미는 데이터가 아니야.** 그래서 `search`, `print`, 이터레이터가 전부 `head.next` 에서 시작해. 교재 360p: **"검색을 시작하는 위치는 head가 아니라 head.next입니다."**

*(답을 적은 뒤 실행)*

In [ ]:
e = DoubleLinkedList()
print("빈 리스트")
print(f"  head.data = {e.head.data}")
print(f"  head.next is head? {e.head.next is e.head}")
print(f"  head.prev is head? {e.head.prev is e.head}")
print(f"  no = {e.no}, len = {len(e)}, is_empty = {e.is_empty()}")
print("\n  → 노드 객체는 1개 있지만 '데이터 노드'는 0개")
print("     no는 '데이터 노드의 개수'만 센다\n")

l = make('A','B')
print("A, B를 넣은 뒤")
print(f"  head.data = {l.head.data}   ← 더미는 그대로 데이터 없음")
print(f"  head.next.data = {l.head.next.data}  ← 진짜 머리")
print(f"  head.prev.data = {l.head.prev.data}  ← 진짜 꼬리")
print("\n  💡 더미는 '리스트의 시작이자 끝'을 표시하는 이정표 역할")

### 6. 🟢 [설명] `is_empty()` 가 이렇게 짧아진 이유

```python
def is_empty(self) -> bool:
    return self.head.next is self.head
```

- 29일차에는 `return self.head is None` 이었어. 오늘은 왜 다를까? ①________________
- 🔥 **오늘 `head` 가 `None` 이 되는 순간이 있을까?** ②____
  → 더미는 **절대 삭제되지 않아.** `remove_current_node` 가 `if not self.is_empty():` 로 막고 있거든.
- 그래서 오늘 코드에는 **`is None` 검사가 한 번도 안 나와.** 확인해봐.
- 이 성질이 왜 좋을까? 29일차 4번에서 `head` 가 `None` 이라 `head.next` 에서 `AttributeError` 가 났던 걸 떠올려봐.

*(답을 적은 뒤 실행)*

In [ ]:
import inspect
src = inspect.getsource(DoubleLinkedList)
none_checks = [ln.strip() for ln in src.splitlines() if 'is None' in ln or 'is not None' in ln]
print(f"오늘 코드의 'is None' 검사 개수: {len(none_checks)}개")
print("→ 단 하나도 없다! 🔥\n")

print("29일차(08-2) 코드와 비교:")
old = """    def add_last(self, data):
        if self.head is None:          # ← 분기
            self.add_first(data)
        else:
            ptr = self.head
            while ptr.next is not None:  # ← 분기
                ptr = ptr.next
            ..."""
print(old)
print("\n오늘(08-4):")
print("""    def add_last(self, data):
        self.current = self.head.prev    # 분기 없음!
        self.add(data)""")
print("\n  → 3줄 vs 2줄. 그리고 while이 사라졌다")

print("\n[더미가 절대 삭제되지 않는 이유]")
e = DoubleLinkedList()
for f, nm in ((e.remove_first,'remove_first'), (e.remove_last,'remove_last'),
              (e.remove_current_node,'remove_current_node')):
    f()
print(f"  빈 리스트에서 삭제 3종 호출 → head 살아있나? {e.head is not None}, no={e.no}")
print("  → if not self.is_empty(): 가 전부 막아준다 ✅")

### 7. 🔴 [그림] 🔥 더미가 없애는 분기 — `add` 하나로 끝

교재 366p:
> **"08-3절에서 배운 연결 리스트 프로그램과는 달리 리스트의 맨 앞에 더미 노드가 있으므로 '빈 리스트에 삽입 처리하는 과정'과 '리스트의 맨 앞에 삽입 처리하는 과정'을 다룰 필요가 없습니다."**

```python
def add(self, data):
    node = Node(data, self.current, self.current.next)
    self.current.next.prev = node
    self.current.next = node
    self.current = node
    self.no += 1

def add_first(self, data):
    self.current = self.head            # 더미 바로 뒤
    self.add(data)

def add_last(self, data):
    self.current = self.head.prev       # 꼬리 바로 뒤
    self.add(data)
```

🔥 **`add_first` 와 `add_last` 가 각각 2줄이야.** 그리고 **분기가 하나도 없어.**

**세 가지 상황을 직접 그려서 확인해봐:**

**(가) 빈 리스트에 `add_first('A')`**
- `current = head` (더미)
- `Node('A', head, head.next)` 인데 `head.next` 는 ①____ (빈 리스트니까)
- 결과: `더미 ⇄ A ⇄ 더미` — 정상 ②____

**(나) `A ⇄ B` 에서 `add_first('X')`**
- `current = head`, `head.next` 는 ③____
- 결과: ④________________

**(다) `A ⇄ B` 에서 `add_last('Y')`**
- `current = head.prev` = ⑤____ (꼬리 B)
- `current.next` 는 ⑥____ (더미!)
- 결과: ⑦________________

- 🔥 **(가)(나)(다)가 전부 같은 `add` 코드로 처리돼.** 29일차는 각각 다른 처리가 필요했지?
- **왜 가능할까?** 한 문장으로: ⑧________________

*(답을 적은 뒤 실행)*

In [ ]:
print("(가) 빈 리스트에 add_first")
l = DoubleLinkedList()
print(f"  삽입 전: current is head? {l.current is l.head}, head.next is head? {l.head.next is l.head}")
l.add_first('A')
draw(l, "  결과")

print("(나) A⇄B 에서 add_first('X')")
l2 = make('A','B')
l2.add_first('X')
draw(l2, "  결과")

print("(다) A⇄B 에서 add_last('Y')")
l3 = make('A','B')
print(f"  current = head.prev = 꼬리 '{l3.head.prev.data}'")
print(f"  current.next = 더미? {l3.head.prev.next is l3.head}")
l3.add_last('Y')
draw(l3, "  결과")

print("🔥 세 경우 모두 같은 add() 코드가 처리했다")
print("   더미 덕분에 'current.next 가 항상 존재'하기 때문 (None일 수 없음)")

### 8. 🟡 [예측] 더미가 없다면?

**사고 실험**: 더미 노드를 없애고 순수 원형 이중 리스트를 만들면 어떤 분기가 부활할까?

```python
# 더미 없는 버전 (head가 진짜 머리 노드를 가리킴)
def add_first(self, data):
    if self.head is None:              # ← 분기 부활!
        node = Node(data)              # 자기 자신을 가리킴
        self.head = node
    else:
        node = Node(data, self.head.prev, self.head)
        self.head.prev.next = node
        self.head.prev = node
        self.head = node               # ← 머리 갱신도 필요
    self.no += 1
```

- 부활하는 분기가 몇 개야? ①____
- `self.head = node` 같은 **머리 갱신**이 왜 추가로 필요해졌지? ②________________
- 반대로 더미가 있으면 `head` 는 **한 번이라도 바뀌나?** ③____
- 🔥 그래서 더미의 진짜 이점을 한 줄로 정리하면: ④________________

*(답을 적은 뒤 실행)*

In [ ]:
print("[더미가 있으면 head가 절대 안 바뀐다]")
l = DoubleLinkedList()
head_id = id(l.head)
for v in 'ABCDE': l.add_last(v)
l.add_first('X')
l.remove_first(); l.remove_last()
l.clear()
for v in 'PQ': l.add_first(v)
print(f"  온갖 조작 후 head가 같은 객체? {id(l.head) == head_id} ✅")
print("  → head를 갱신하는 코드가 아예 필요 없다\n")

print("[29일차(더미 없음)와 비교]")
print("  08-2 add_first:  self.head = self.current = Node(data, ptr)   ← head 갱신")
print("  08-2 remove_first: self.head = self.current = self.head.next  ← head 갱신")
print("  08-4 add_first:  self.current = self.head; self.add(data)     ← head 안 건드림")
print("\n  💡 '경계(맨 앞/맨 뒤)'가 특별 취급을 안 받게 만드는 게 더미의 핵심")
print("     이런 기법을 sentinel(보초) 노드라고도 부른다")

### 9. 🟡 [설명] 이터레이터도 더미 덕을 본다

```python
class DoubleLinkedListIterator:
    def __init__(self, head):
        self.head = head
        self.current = head.next        # ① 더미 다음부터
    def __next__(self):
        if self.current is self.head:   # ② 한 바퀴 돌면 종료
            raise StopIteration
        ...
```

- 29일차 이터레이터의 종료 조건은 `if self.current is None:` 이었어. 오늘은? ①________________
- 🔥 **`__reversed__` 가 새로 생겼어.** 29일차에는 못 만들었지. 왜 이제 가능해?  ②________________
- `reversed(lst)` 를 쓰려면 클래스에 뭘 구현해야 해? ③____
- 역순 이터레이터는 어디서 시작해? ④____ (`head.prev`)

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C','D')
print("정방향:", *[e for e in l])
print("역방향:", *[e for e in reversed(l)])
print()
print("[중첩 순회도 가능]")
print("  ", [(a, b) for a in l for b in reversed(l)][:6], "...")
print("  → 이터레이터를 매번 새로 만드니 서로 간섭하지 않는다 (29일차 18번)\n")

print("[current를 건드리지 않는가?]")
l.search('B')
print(f"  search('B') 후 current = {l.current.data}")
for _ in l: pass
for _ in reversed(l): pass
print(f"  양방향 순회 후 current = {l.current.data}  ← 그대로 ✅")

print("\n[29일차에는 왜 역순 순회가 불가능했나]")
print("  08-2 Node에는 next만 있고 prev가 없었다")
print("  → 뒤에서 앞으로 갈 방법 자체가 없음")
print("  → 역순으로 출력하려면 전부 리스트에 담아 뒤집어야 했다 (O(n) 추가 메모리)")

---
# ✏️ PART 3 — 삽입·삭제를 그림으로 (10~13번)

> 29일차 PART 3의 원칙이 오늘도 통해: **"끊기 전에 먼저 이어라."**
> 다만 이제 화살표가 **양방향**이라 신경 쓸 게 2배야.

### 10. 🟡 [그림] `add()` — 교재 그림 8-29

교재 365p. `A ⇄ B ⇄ C` 에서 `current = B` 일 때 `add('D')` 를 실행해.

```python
node = Node(data, self.current, self.current.next)   # 1
self.current.next.prev = node                        # 2
self.current.next = node                             # 3
self.current = node                                  # 4
self.no += 1
```

**직접 그려봐:**

**단계 1** — `Node('D', current, current.next)` 생성
```
                    [D]
                   ↙   ↘
              prev=①    next=②
   [A] ⇄ [B] ⇄ [C]
```
- ① = ____ , ② = ____
- 이 시점에서 **A, B, C는 아직 D를 모르지?** D만 일방적으로 B와 C를 가리키고 있어.

**단계 2** — `current.next.prev = node`
- `current.next` 는 ③____ 이니, 결국 ④________ 를 실행하는 것
- 이제 C가 D를 **앞쪽으로** 가리킴

**단계 3** — `current.next = node`
- ⑤________ 를 실행. 이제 B가 D를 **뒤쪽으로** 가리킴

**최종**: `A ⇄ B ⇄ D ⇄ C`

🔥 **단계 2와 3의 순서를 바꾸면 어떻게 될까?**
- 먼저 `current.next = node` 를 하면, 그다음 `current.next.prev` 는 **누구의 prev**가 되지? ⑥____
- 이게 17번(보충수업 8-4)의 핵심이야. 지금은 예측만 해둬.

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
l.search('B')
print(f"삽입 전 (current = {l.current.data})")
draw(l)

cur = l.current
node = Node('D', cur, cur.next)
print("단계 1: Node('D', current, current.next) 생성")
print(f"  D.prev.data = {node.prev.data}, D.next.data = {node.next.data}")
print(f"  하지만 B.next 는 아직 {cur.next.data} (D를 모름)")
print(f"  C.prev 도 아직 {cur.next.prev.data} (D를 모름)\n")

cur.next.prev = node
print(f"단계 2: current.next.prev = node  →  C.prev = D")
cur.next = node
print(f"단계 3: current.next = node       →  B.next = D")
l.current = node; l.no += 1
draw(l, "\n최종")

### 11. 🟡 [그림] `remove_current_node()` — 교재 그림 8-31

교재 368p. `A ⇄ B ⇄ C` 에서 `current = B` 를 삭제.

```python
if not self.is_empty():
    self.current.prev.next = self.current.next    # 1
    self.current.next.prev = self.current.prev    # 2
    self.current = self.current.prev              # 3
    self.no -= 1
    if self.current is self.head:                 # 4
        self.current = self.head.next
```

**직접 그려봐:**
- 단계 1: `current.prev` = ①____ , `current.next` = ②____ → 결국 ③________ 실행
- 단계 2: ④________ 실행
- 이제 B를 가리키는 화살표가 **몇 개** 남았어? ⑤____
- 단계 3: `current` 가 ⑥____ 로 이동

🔥 **단계 4가 왜 필요할까?**
- **머리 노드를 삭제**하면 `current.prev` 는 뭐가 되지? ⑦____
- 더미가 주목 노드가 되면 `print_current_node()` 가 ⑧________ 를 출력하게 돼. 그래서 보정하는 거야.
- 29일차 17번의 원리 **"마지막으로 손댄, 리스트에 남아 있는 노드"** 를 여기 적용하면? 더미는 "데이터 노드"가 아니니 **머리로 밀어주는** 거지.

**세 경우를 각각 예측해봐:**
| 삭제한 노드 | 삭제 후 current |
|---|---|
| 머리 A | ⑨ |
| 중간 B | ⑩ |
| 꼬리 C | ⑪ |

*(예측을 적은 뒤 실행)*

In [ ]:
for pos, label in ((0,'머리 A'), (1,'중간 B'), (2,'꼬리 C')):
    l = make('A','B','C')
    p = l.head.next
    for _ in range(pos): p = p.next
    l.current = p
    before = l.current.data
    l.remove_current_node()
    cur = '더미' if l.current is l.head else l.current.data
    print(f"  {label} 삭제 → current = {cur}")

print("\n[단계 4가 없으면?]")
class NoFix(DoubleLinkedList):
    def remove_current_node(self):
        if not self.is_empty():
            self.current.prev.next = self.current.next
            self.current.next.prev = self.current.prev
            self.current = self.current.prev
            self.no -= 1
            # if self.current is self.head: ... 를 뺐다

n = NoFix()
for v in 'ABC': n.add_last(v)
n.current = n.head.next          # 머리 A
n.remove_current_node()
print(f"  머리 삭제 후 current is head(더미)? {n.current is n.head}")
n.print_current_node()
print("  → 데이터가 없는 더미를 주목하게 되어 None이 출력된다 💥")

print("\n[마지막 하나를 지우면?]")
one = make('A')
one.current = one.head.next
one.remove_current_node()
print(f"  is_empty={one.is_empty()}, current is head? {one.current is one.head}, no={one.no}")
print("  → 이때는 보정해도 head.next가 더미 자신이라 current=더미가 맞다 ✅")

### 12. 🟢 [설명] `add_first`/`add_last`/`remove_first`/`remove_last` 는 전부 2줄

```python
def add_first(self, data):   self.current = self.head;      self.add(data)
def add_last(self, data):    self.current = self.head.prev; self.add(data)
def remove_first(self):      self.current = self.head.next; self.remove_current_node()
def remove_last(self):       self.current = self.head.prev; self.remove_current_node()
```

**표를 채워봐** — 각 함수가 `current` 를 어디로 옮기는지:

| 함수 | `current` 를 어디로 | 왜 |
|---|---|---|
| `add_first` | ① `head`(더미) | ② |
| `add_last` | ③ | ④ |
| `remove_first` | ⑤ | ⑥ |
| `remove_last` | ⑦ | ⑧ |

- 🔥 `add_first` 는 **더미 자신**을, `remove_first` 는 **더미의 다음**을 가리켜. 왜 다를까? ⑨________________
  💡 힌트: `add` 는 "current **바로 뒤**에 삽입", `remove_current_node` 는 "current **자신**을 삭제"
- `add_last` 와 `remove_last` 는 둘 다 `head.prev` 야. 이건 왜 같아? ⑩________________

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
print("add는 'current 바로 뒤'에 삽입한다")
print(f"  add_first → current = head(더미) → 더미 뒤 = 맨 앞 ✅")
print(f"  add_last  → current = head.prev(꼬리 C) → 꼬리 뒤 = 맨 끝 ✅\n")
print("remove_current_node는 'current 자신'을 삭제한다")
print(f"  remove_first → current = head.next(머리 A) → A 삭제 ✅")
print(f"  remove_last  → current = head.prev(꼬리 C) → C 삭제 ✅\n")

print("[실제 동작 확인]")
for op in ('add_first', 'add_last', 'remove_first', 'remove_last'):
    t = make('A','B','C')
    if op.startswith('add'): getattr(t, op)('X')
    else: getattr(t, op)()
    vals = [e for e in t]
    cur = '더미' if t.current is t.head else t.current.data
    print(f"  {op:14s} → {vals}, current={cur}")

### 13. 🔴 [설명] `remove(p)` 는 왜 O(n)일까

```python
def remove(self, p: Node) -> None:
    ptr = self.head.next
    while ptr is not self.head:          # ← 리스트 전체를 훑는다
        if ptr is p:
            self.current = p
            self.remove_current_node()
            break
        ptr = ptr.next
```

🔥 **이상하지 않아?** 이중 연결 리스트면 `p.prev` 와 `p.next` 를 알고 있으니 **탐색 없이 바로** 지울 수 있잖아:

```python
def remove_fast(self, p):
    p.prev.next = p.next
    p.next.prev = p.prev
    self.no -= 1
    ...
```

- 이러면 시간 복잡도가 ①____ 가 돼.
- **그럼 교재는 왜 굳이 while로 훑을까?** 이유를 생각해봐: ②________________
  💡 힌트: `p` 가 **이 리스트에 없는 노드**라면? 다른 리스트의 노드이거나, 이미 삭제된 노드라면?
- 🔥 **트레이드오프**를 정리해봐:

| | 교재 방식 (탐색) | 빠른 방식 (직접) |
|---|---|---|
| 시간 복잡도 | ③ | ④ |
| `p` 가 없는 노드일 때 | ⑤ | ⑥ |
| 안전성 | ⑦ | ⑧ |

- 실무에서는 어느 쪽을 쓸까? 상황에 따라 다른데, **호출자를 신뢰할 수 있으면** 빠른 쪽을 쓰지.
- 29일차 11번의 `remove(p)` 도 `while` 로 앞쪽 노드를 찾았어. 그건 **어쩔 수 없었지만**(prev가 없으니), 오늘은 **선택**이야.

*(답을 적은 뒤 실행)*

In [ ]:
class FastRemove(DoubleLinkedList):
    def remove(self, p: Node) -> None:
        if p is self.head: return          # 더미는 못 지움
        p.prev.next = p.next
        p.next.prev = p.prev
        self.no -= 1
        self.current = p.prev
        if self.current is self.head:
            self.current = self.head.next

print("[결과가 같은가?]")
for cls, nm in ((DoubleLinkedList,'교재(탐색)'), (FastRemove,'빠른(직접)')):
    t = cls()
    for v in 'ABCDE': t.add_last(v)
    q = t.head.next.next     # C
    t.remove(q)
    print(f"  {nm:12s} → {[e for e in t]}, no={t.no}")

print("\n[리스트에 없는 노드를 지우려 하면?]")
outsider = Node('Z')
outsider.prev = outsider; outsider.next = outsider
for cls, nm in ((DoubleLinkedList,'교재(탐색)'), (FastRemove,'빠른(직접)')):
    t = cls()
    for v in 'ABC': t.add_last(v)
    t.remove(outsider)
    print(f"  {nm:12s} → {[e for e in t]}, no={t.no}  {'✅ 안전' if t.no==3 else '❌ no가 깨짐!'}")

print("\n[속도 비교 — 꼬리 근처 노드를 반복 삭제]")
import random
for n in (2000, 4000):
    for cls, nm in ((DoubleLinkedList,'교재'), (FastRemove,'빠른')):
        t = cls()
        for i in range(n): t.add_last(i)
        nodes = []
        p = t.head.next
        while p is not t.head: nodes.append(p); p = p.next
        targets = nodes[-200:]
        st = time.perf_counter()
        for q in targets: t.remove(q)
        el = time.perf_counter() - st
        print(f"  n={n} {nm:4s}: {el*1000:7.2f}ms", end="  |  " if nm=='교재' else "\n")

---
# 💻 PART 4 — 코드 구현 (14~18번)

### 14. 🟢 [빈칸] `Node` 와 `__init__`

R-2의 `or` 연산자, 5번의 더미 노드가 여기 들어가.

**기대 출력**
```
Node(): prev is self? True, next is self? True
빈 리스트: no=0, is_empty=True, head.data=None
head.next is head? True
```

In [ ]:
class MyNode:
    """원형 이중 연결 리스트용 노드 클래스"""
    def __init__(self, data: Any = None, prev: MyNode = None, next: MyNode = None) -> None:
        self.data = ___              # ①
        self.prev = ___              # ② 인자가 없으면 자기 자신! (R-2)
        self.next = ___              # ③


class MyDoubleLinkedList:
    def __init__(self) -> None:
        self.head = self.current = ___   # ④ 더미 노드를 만든다 (5번)
        self.no = 0

    def __len__(self) -> int:
        return ___                   # ⑤

    def is_empty(self) -> bool:
        return ___                   # ⑥ 더미가 자기 자신을 가리키면 빈 리스트 (6번)


n = MyNode()
print(f"Node(): prev is self? {n.prev is n}, next is self? {n.next is n}")
l = MyDoubleLinkedList()
print(f"빈 리스트: no={l.no}, is_empty={l.is_empty()}, head.data={l.head.data}")
print(f"head.next is head? {l.head.next is l.head}")

### 15. 🟡 [빈칸] `search` 와 이동 함수

🔥 **29일차와 결정적으로 다른 두 곳**: 시작 위치와 종료 조건.

**기대 출력**
```
search('C') = 2, current = C
search('Z') = -1
next 3번: A → B → C → D, 4번째 False
prev 3번: D → C → B → A, 4번째 False
```

In [ ]:
def my_search(self, data: Any) -> Any:
    cnt = 0
    ptr = ___                        # ① 어디서 출발? (더미가 아니다!)
    while ___:                       # ② 언제까지? (None이 아니다!)
        if data == ptr.data:
            self.current = ptr
            return cnt
        cnt += 1
        ptr = ___                    # ③
    return ___                       # ④


def my_next(self) -> bool:
    if self.is_empty() or ___:       # ⑤ 뒤쪽이 더미면 = 꼬리면 못 감
        return False
    self.current = ___               # ⑥
    return True


def my_prev(self) -> bool:
    if self.is_empty() or ___:       # ⑦ 앞쪽이 더미면 = 머리면 못 감
        return False
    self.current = ___               # ⑧
    return True


MyDoubleLinkedList.search = my_search
MyDoubleLinkedList.next = my_next
MyDoubleLinkedList.prev = my_prev
MyDoubleLinkedList.add = DoubleLinkedList.add
MyDoubleLinkedList.add_last = DoubleLinkedList.add_last

lst = MyDoubleLinkedList()
for v in 'ABCD': lst.add_last(v)
print(f"search('C') = {lst.search('C')}, current = {lst.current.data}")
print(f"search('Z') = {lst.search('Z')}")

lst.search('A')
p = [lst.current.data]
while lst.next(): p.append(lst.current.data)
print(f"next 3번: {' → '.join(p)}, 4번째 {lst.next()}")

lst.search('D')
p = [lst.current.data]
while lst.prev(): p.append(lst.current.data)
print(f"prev 3번: {' → '.join(p)}, 4번째 {lst.prev()}")

### 16. 🔴 [빈칸] `add` 계열 — 오늘의 핵심

10번의 그림 그대로. **순서가 생명이야** (17번에서 왜인지 밝혀져).

**기대 출력**
```
add_first: ['X', 'A', 'B']
add_last : ['X', 'A', 'B', 'Y']
빈 리스트에 add_first: ['Z']
역방향도 정상: ['Y', 'B', 'A', 'X']
```

In [ ]:
def my_add(self, data: Any) -> None:
    """주목 노드 바로 뒤에 노드를 삽입"""
    node = MyNode(data, ___, ___)    # ①② 새 노드의 prev와 next
    ___ = node                       # ③ 뒤쪽 노드의 prev를 갱신 (먼저!)
    ___ = node                       # ④ current의 next를 갱신 (나중!)
    self.current = node
    self.no += 1


def my_add_first(self, data: Any) -> None:
    self.current = ___               # ⑤ 더미 (그 '바로 뒤'가 맨 앞)
    self.add(data)


def my_add_last(self, data: Any) -> None:
    self.current = ___               # ⑥ 꼬리 (그 '바로 뒤'가 맨 끝) — O(1)!
    self.add(data)


MyDoubleLinkedList.add = my_add
MyDoubleLinkedList.add_first = my_add_first
MyDoubleLinkedList.add_last = my_add_last

def fwd(l):
    r, p = [], l.head.next
    while p is not l.head: r.append(p.data); p = p.next
    return r
def bwd(l):
    r, p = [], l.head.prev
    while p is not l.head: r.append(p.data); p = p.prev
    return r

lst = MyDoubleLinkedList()
for v in 'AB': lst.add_last(v)
lst.add_first('X'); print(f"add_first: {fwd(lst)}")
lst.add_last('Y');  print(f"add_last : {fwd(lst)}")
e = MyDoubleLinkedList(); e.add_first('Z')
print(f"빈 리스트에 add_first: {fwd(e)}")
print(f"역방향도 정상: {bwd(lst)}")

### 17. 🔴 [디버깅] 🔥🔥 보충수업 8-4 — 한 줄로 합치면 안 되는 이유

교재 370p가 이걸 콕 집어 경고해:

```python
# 파이썬에서는 제대로 동작하지 않는 문법 (C, Java 등에서는 제대로 동작)
self.current.next = self.current.next.prev = node
```

**왜 C/Java에서는 되는데 파이썬에서는 안 될까?**

교재 371p:
> **"C 언어, Java 등의 언어에서는 대입 연산자 '='은 오른쪽 결합 연산자입니다."** → 오른쪽부터 해석:
> ```
> self.current.next.prev = node    # 대입 [1]
> self.current.next = node         # 대입 [2]
> ```
>
> **"그런데 파이썬에서는 위와 같은 연속 대입은 다음처럼 수행됩니다(순서가 거꾸로입니다)."**
> ```
> self.current.next = node         # 대입 [X]  ← 먼저!
> self.current.next.prev = node    # 대입 [Y]
> ```

🔥 **[Y]가 실행될 때 `self.current.next` 는 이미 `node` 야.** 그럼 `node.prev = node` 가 되어 **자기 자신을 가리켜.**

**예측해봐:**
- 정상: `A ⇄ B ⇄ C` 에 D를 넣으면 → `A ⇄ B ⇄ D ⇄ C`
- 한 줄 버전: 정방향은 어떻게 될까? ①________________
- 역방향은? ②________________
- 에러가 날까, 조용히 깨질까? ③____

*(예측을 적은 뒤 실행 — 결과가 꽤 충격적이야)*

In [ ]:
class BadAdd(DoubleLinkedList):
    def add(self, data):
        node = Node(data, self.current, self.current.next)
        self.current.next = self.current.next.prev = node   # 🐛 한 줄
        self.current = node
        self.no += 1

for cls, nm in ((DoubleLinkedList, '교재 (2줄)'), (BadAdd, '한 줄 연속대입')):
    l = cls()
    for v in 'ABC': l.add_last(v)
    print(f"{nm}:")
    draw(l)

print("[일반 변수로 최소 재현]")
class Box:
    def __init__(s, n): s.name = n; s.link = None
    def __repr__(s): return s.name

a = Box('a'); b = Box('b'); a.link = b; new = Box('new')
a.link = a.link.link = new
print(f"  한 줄:  a.link={a.link}, new.link={new.link}, b.link={b.link}")
print("          → b.link가 안 바뀌고 new.link가 자기 자신을 가리킴 ❌")

a2 = Box('a'); b2 = Box('b'); a2.link = b2; new2 = Box('new')
a2.link.link = new2      # 먼저 뒤쪽을 잇고
a2.link = new2           # 나중에 앞쪽을 잇는다
print(f"  두 줄:  a2.link={a2.link}, new2.link={new2.link}, b2.link={b2.link}")
print("          → 정상 ✅\n")

print("[규칙]  파이썬의 a = b = expr 은:")
print("   1) expr 을 한 번만 평가하고")
print("   2) 왼쪽부터 차례로 대입한다  ← C/Java와 반대!")
print("\n💡 26일차 2번의 'pl, pr = left, right = range.pop()' 과 같은 문법인데,")
print("   거기서는 대상이 단순 변수라 순서가 문제되지 않았다.")
print("   오늘은 대입 대상이 '다른 대입의 결과에 의존'해서 터진 것.")

### 18. 🔴 [빈칸] `remove` 계열과 이터레이터

11~13번의 그림 그대로.

**기대 출력**
```
remove_first: ['B', 'C', 'D']
remove_last : ['B', 'C']
머리 삭제 후 current: B
정방향: A B C / 역방향: C B A
```

In [ ]:
def my_remove_current_node(self) -> None:
    if not self.is_empty():
        ___ = self.current.next      # ① 앞쪽 노드의 next를 뒤쪽으로
        ___ = self.current.prev      # ② 뒤쪽 노드의 prev를 앞쪽으로
        self.current = ___           # ③ 주목을 앞쪽으로 이동
        self.no -= 1
        if ___:                      # ④ 더미를 주목하게 됐다면
            self.current = ___       # ⑤ 머리로 보정


def my_remove_first(self) -> None:
    self.current = ___               # ⑥ 머리 노드
    self.remove_current_node()


def my_remove_last(self) -> None:
    self.current = ___               # ⑦ 꼬리 노드
    self.remove_current_node()


class MyIterator:
    def __init__(self, head):
        self.head = head
        self.current = ___           # ⑧ 더미 다음부터
    def __iter__(self): return self
    def __next__(self):
        if ___:                      # ⑨ 한 바퀴 돌면 종료
            raise StopIteration
        data = self.current.data
        self.current = ___           # ⑩
        return data


class MyReverseIterator:
    def __init__(self, head):
        self.head = head
        self.current = ___           # ⑪ 더미 앞부터 = 꼬리부터
    def __iter__(self): return self
    def __next__(self):
        if self.current is self.head:
            raise StopIteration
        data = self.current.data
        self.current = ___           # ⑫
        return data


MyDoubleLinkedList.remove_current_node = my_remove_current_node
MyDoubleLinkedList.remove_first = my_remove_first
MyDoubleLinkedList.remove_last = my_remove_last
MyDoubleLinkedList.__iter__ = lambda self: MyIterator(self.head)
MyDoubleLinkedList.__reversed__ = lambda self: MyReverseIterator(self.head)

lst = MyDoubleLinkedList()
for v in 'ABCD': lst.add_last(v)
lst.remove_first(); print(f"remove_first: {fwd(lst)}")
lst.remove_last();  print(f"remove_last : {fwd(lst)}")

t = MyDoubleLinkedList()
for v in 'ABC': t.add_last(v)
t.search('A'); t.remove_current_node()
print(f"머리 삭제 후 current: {t.current.data}")

t2 = MyDoubleLinkedList()
for v in 'ABC': t2.add_last(v)
print(f"정방향: {' '.join(str(e) for e in t2)} / 역방향: {' '.join(str(e) for e in reversed(t2))}")

---
# 📐 PART 5 — 성능과 총정리 (19~21번)

### 19. 🟡 [실험] 🔥 `add_last` 가 정말 O(1)인가

29일차 21번에서 `add_last` 로 n개를 넣으면 **O(n²)** 이었어. 오늘은?

**예측해봐** (n을 2배로 늘리면):
- 08-2 (29일차) 소요 시간: ①____ 배
- 08-4 (오늘) 소요 시간: ②____ 배

*(예측을 적은 뒤 실행)*

In [ ]:
class OldNode:
    def __init__(s, d=None, n=None): s.data = d; s.next = n
class OldList:
    """29일차 08-2 방식"""
    def __init__(s): s.no = 0; s.head = None; s.current = None
    def add_last(s, d):
        if s.head is None:
            s.head = s.current = OldNode(d, None); s.no += 1
        else:
            p = s.head
            while p.next is not None: p = p.next
            p.next = s.current = OldNode(d, None); s.no += 1

print("      n | 08-2 (29일차) | 08-4 (오늘) |  배수")
print("  " + "-"*48)
for n in (500, 1000, 2000, 4000):
    t = time.perf_counter()
    o = OldList()
    for i in range(n): o.add_last(i)
    e1 = time.perf_counter() - t
    t = time.perf_counter()
    d = DoubleLinkedList()
    for i in range(n): d.add_last(i)
    e2 = time.perf_counter() - t
    print(f"  {n:5d} | {e1*1000:10.2f}ms | {e2*1000:8.2f}ms | {e1/e2:5.1f}배")

print("\n🔥 08-2: n이 2배면 시간은 약 4배 → O(n²)")
print("   08-4: n이 2배면 시간도 약 2배 → O(n), 즉 1회 삽입이 O(1) ✅")
print("\n[왜?]  head.prev 가 곧 꼬리이므로 while 탐색이 사라졌다")

### 20. 🟢 [설명] `clear()` 의 `self.no = 0` 은 필요할까

```python
def clear(self) -> None:
    while not self.is_empty():
        self.remove_first()
    self.no = 0            # ← 이 줄
```

- `remove_first` 가 `no -= 1` 을 제대로 하고 있다면, 루프가 끝났을 때 `no` 는 이미 ①____ 이어야 해.
- 그럼 이 줄은 ②________________ 야.
- 🔥 **29일차 20번**을 떠올려봐. 거기서는 `remove_first` 의 `no -= 1` 이 `if` **바깥**에 있어서 `no` 가 음수가 됐지. 오늘 코드는?
- 30일차 `clear()` 에는 `self.no = 0` 이 **없었어.** 08-3와 08-4 중 어느 쪽이 더 일관적일까?

*(답을 적은 뒤 실행)*

In [ ]:
print("[remove_first 반복만으로 no가 0이 되는가?]")
l = make('A','B','C','D','E')
print(f"  시작 no = {l.no}")
while not l.is_empty():
    l.remove_first()
print(f"  반복 후 no = {l.no}   ← self.no = 0 없이도 0 ✅")

print("\n[세 버전의 clear() 비교]")
print("  08-2 (29일차): self.no = 0 있음 — 하지만 remove_first에 버그가 있어서 필요했음")
print("  08-3 (30일차): self.no = 0 없음 — remove_first가 정상이라 불필요")
print("  08-4 (오늘)  : self.no = 0 있음 — 정상이지만 안전장치로 남겨둠")

print("\n[빈 리스트에서 삭제해도 안전한가?]")
e = DoubleLinkedList()
for _ in range(3):
    e.remove_first(); e.remove_last(); e.remove_current_node()
print(f"  9번 호출 후 no = {e.no}, len = {len(e)} ✅")
print("  → if not self.is_empty(): 가 전부 막는다 (29일차 20번 버그 없음)")

### 21. 🟢 [정리] 08장 완전 총정리

08장을 다 배웠어. **네 가지 리스트 구현**을 비교해봐.

| | 08-2 포인터 (29일) | 08-3 커서 (30일) | 08-4 원형이중 (오늘) |
|---|---|---|---|
| 노드의 정체 | 객체 | 배열 원소 | ① |
| "없음" 표시 | `None` | `Null(-1)` | ② **더미 노드** |
| `add_first` | O(1) | O(1) | ③ |
| `add_last` | O(n) | O(n) | ④ |
| `remove_last` | O(n) | O(n) | ⑤ |
| 앞쪽으로 이동 | ❌ | ❌ | ⑥ |
| 역순 순회 | ❌ | ❌ | ⑦ |
| 크기 제한 | 없음 | ⑧ | ⑨ |
| 빈 리스트 분기 | 많음 | 많음 | ⑩ |
| 노드당 메모리 | data + next | data + next + dnext | ⑪ |

**최종 질문 4개**
1. 오늘 구조가 이전 둘보다 **모든 면에서 나을까?** 대가는 뭐지?
2. 더미 노드가 없애준 것을 **두 가지**로 정리하면?
3. 파이썬 `collections.deque` 가 바로 이 구조야. `deque` 의 `appendleft`, `pop` 이 O(1)인 이유를 오늘 배운 걸로 설명해봐.
4. 29일차 3번의 "비상 연락망" 비유를 오늘 구조에 맞게 바꾸면?

*(답을 적은 뒤 실행)*

In [ ]:
import sys
from collections import deque

print("[메모리 비교 — 노드당 필드 수]")
print("  08-2: data, next          (2개)")
print("  08-3: data, next, dnext   (3개) + 배열 슬롯")
print("  08-4: data, prev, next    (3개)")
n = 1000
l = make(*range(n))
node = l.head.next
sz = sys.getsizeof(node) + sys.getsizeof(node.__dict__)
print(f"\n  Node 1개 ≈ {sz} 바이트, {n}개 ≈ {sz*n/1024:.1f} KB")
print(f"  파이썬 list {n}개 = {sys.getsizeof(list(range(n)))/1024:.1f} KB")

print("\n[collections.deque 와 비교]")
d = deque(range(n))
print(f"  deque {n}개 = {sys.getsizeof(d)/1024:.1f} KB")
for op, our, dq in (("맨 앞 삽입", lambda: l.add_first(-1), lambda: d.appendleft(-1)),
                    ("맨 끝 삽입", lambda: l.add_last(-1),  lambda: d.append(-1)),
                    ("맨 앞 삭제", lambda: l.remove_first(), lambda: d.popleft()),
                    ("맨 끝 삭제", lambda: l.remove_last(),  lambda: d.pop())):
    t = time.perf_counter()
    for _ in range(2000): our()
    e1 = time.perf_counter() - t
    t = time.perf_counter()
    for _ in range(2000): dq()
    e2 = time.perf_counter() - t
    print(f"  {op}: 우리 구현 {e1*1000:6.2f}ms | deque {e2*1000:5.2f}ms  ({e1/e2:,.0f}배)")

print("\n💡 deque도 내부적으로 양방향 구조 (정확히는 블록 이중 연결 리스트)")
print("   네 방향 연산이 모두 O(1)인 이유가 오늘 배운 그 원리다")
print("   C로 구현되어 있어 속도만 수십 배 빠를 뿐, 발상은 같다")

---
---

# ✅ 정답 & 해설

> ⚠️ **화살표를 직접 그린 뒤에 내려와.** 특히 4·10·11번은 손으로 그려야 남아.

---

## 🔁 Remind

### R-1
- (가) ① **꼬리 노드를 찾는다** ② **O(n)**
- (나) ③ **꼬리를 지우려면 그 앞쪽 노드의 `next` 를 끊어야 하는데, `next` 만 있는 구조에서는 앞으로 되돌아갈 수 없어서** 한 칸 뒤처져 따라오는 커서가 필요했어
- (다) ④ **6개** (`add_last`, `remove_first`, `remove_last`, `remove`, `clear`, `next`) — 거의 모든 함수에 있었지

### R-2
- ① **`self`** (자기 자신) ② **그 노드 그대로**
- ③ `self.prev = prev if prev is not None else self`

🔥 이 한 줄이 **원형의 씨앗**이야. 노드를 혼자 만들면 자기 자신을 가리키는 **길이 1짜리 원형**이 되거든. 그래서 `__init__` 에서 `Node()` 하나만 만들면 곧바로 유효한 빈 원형 리스트가 완성돼.

---

## 🎯 PART 1 해설

### 1. 원형 리스트
- ① **머리 노드 `A`** (`None` 이 아님)
- ② **"한 바퀴 돌아 시작점으로 돌아왔는가"** — `ptr is head` 로 판정
- ③ **`head.prev`** 가 곧 꼬리 (이중이면). 단방향 원형이면 여전히 훑어야 해.
- **고리 모양 데이터 예**: 라운드 로빈 스케줄링(프로세스를 순환하며 CPU 배분), 캐러셀/슬라이드쇼, 멀티플레이어 게임의 턴 순서, 원형 버퍼

### 2. 이중 연결 리스트
- ① **`data`** ② **`prev`** ③ **`next`**
- ④ **앞쪽 노드를 찾기 어렵다** (29일차 3번: "비상 연락망에서 뒤돌아 앞 사람에게 연락할 수 없다")
- ⑤ **`p.prev` 가 곧 앞쪽 노드**라서. 따라다닐 필요가 없어졌어.

### 3. 원형 + 이중
- ① **더미 노드** ② **꼬리 노드 `C`**
- ③ **꼬리 노드** ④ **머리 노드**

**실측**: `next` 만 7번 따라가도, `prev` 만 7번 따라가도 더미로 돌아와. 양방향 모두 원형이야.

### 4. 참조식 표
- ① `a.prev` ② `b.prev.prev`
- ③ `e.next.next` ④ `b.prev`
- ⑤ `a.next` (또는 `head.next.next`) ⑥ `d.prev.prev`
- ⑦ `a.next.next` ⑧ `e.prev.prev`
- ⑨ `e.prev` ⑩ `head.prev.prev`
- ⑪ `d.next` ⑫ `a.prev.prev`

**위치 판별식**
- ⑬ **머리 노드** ⑭ **맨 앞에서 2번째** ⑮ **꼬리 노드** ⑯ **맨 끝에서 2번째**

🔥 **`a.prev.prev` 가 `E`(꼬리)** 라는 게 원형의 묘미야. 머리에서 앞으로 두 칸 가면 더미를 지나 꼬리로 넘어가거든.

---

## 👻 PART 2 해설

### 5. 더미 노드
- ① **`None`** (데이터가 없음)
- ② **둘 다 자기 자신** (`prev or self`, `next or self` 의 결과)
- ③ **모순이 아니야.** `no` 는 **데이터 노드의 개수**만 세. 더미는 구조를 유지하기 위한 이정표일 뿐 데이터가 아니거든.

> 🔑 더미는 **"리스트의 시작이자 끝"을 표시하는 표지판**이야. 원형이라 시작과 끝이 같은 지점이 되지.

---

### 6. `is_empty()`
- ① **더미가 항상 존재하므로 `head` 자체는 절대 `None` 이 되지 않아.** 대신 "더미 말고 다른 노드가 있는가"를 물어야 해.
- ② **없어.** 더미는 삭제되지 않아.

**실측**: 오늘 코드에 `is None` 검사가 **0개.** 29일차에는 6개 함수에 있었지.

- **왜 좋은가**: 29일차 4번에서 `head` 가 `None` 이라 `head.next` 에서 `AttributeError` 가 났었지. 오늘은 **`head.next` 가 항상 유효한 노드**라 그런 사고가 원천 차단돼.

---

### 7. 🔥 더미가 없애는 분기
- ① **`head` 자신** (빈 리스트에서 `head.next is head`)
- ② **정상**
- ③ **노드 A** ④ **`더미 ⇄ X ⇄ A ⇄ B ⇄ 더미`**
- ⑤ **노드 B** ⑥ **더미** ⑦ **`더미 ⇄ A ⇄ B ⇄ Y ⇄ 더미`**
- ⑧ **더미 덕분에 `current.next` 가 항상 존재해서** — `None` 을 만날 일이 없으니 "리스트가 비었나", "맨 앞인가"를 따로 물을 필요가 없어.

> 🔑 **경계를 특별 취급하지 않게 만드는 것** — 이게 더미(sentinel) 노드의 본질이야.

---

### 8. 더미가 없다면
- ① **최소 2개** (`head is None` 검사 + 그에 따른 분기)
- ② **머리 노드가 바뀌면 `head` 변수도 따라 바꿔야 하니까.** 더미가 있으면 진짜 머리는 `head.next` 라서 `head` 자체는 손댈 일이 없어.
- ③ **한 번도 안 바뀌어.**

**실측**: `add_last` ×5, `add_first`, `remove_first`, `remove_last`, `clear`, `add_first` ×2 를 다 해도 `id(head)` 가 그대로야.

- ④ **더미의 이점**: **"맨 앞/맨 뒤라는 경계가 사라져서, 빈 리스트 분기와 head 갱신이 모두 불필요해진다."**

💡 이런 기법을 **sentinel(보초) 노드**라고도 불러. 자료구조 전반에서 자주 쓰이는 패턴이야.

---

### 9. 이터레이터
- ① **`if self.current is self.head:`** — `None` 대신 **더미로 돌아왔는지**로 판정
- ② **`prev` 포인터가 생겼기 때문.** 뒤에서 앞으로 갈 수 있어야 역순 순회가 가능하지.
- ③ **`__reversed__`**
- ④ **`head.prev`** (= 꼬리)

**실측**: 정방향/역방향 모두 정상이고, 양방향 순회 후에도 `current` 는 그대로야 (29일차 19번의 원칙 유지).

---

## ✏️ PART 3 해설

### 10. `add()` — 그림 8-29
- ① **`current`(B)** ② **`current.next`(C)**
- ③ **`C`** ④ **`C.prev = D`**
- ⑤ **`B.next = D`**
- ⑥ 🔥 **`node.prev`(= D 자신의 prev)** — 이게 17번의 함정이야!

**단계별 관찰**
```
단계 1 후: D는 B와 C를 알지만, B와 C는 D를 모른다 (일방통행)
단계 2 후: C.prev = D  (C가 D를 앎)
단계 3 후: B.next = D  (B가 D를 앎)
```

> 🔑 **"새 노드가 먼저 이웃을 가리키고, 그다음 이웃이 새 노드를 가리킨다."**
> 29일차의 "끊기 전에 먼저 이어라" 와 같은 정신이야.

---

### 11. `remove_current_node()` — 그림 8-31
- ① **`A`** ② **`C`** ③ **`A.next = C`** ④ **`C.prev = A`**
- ⑤ **0개** — 리스트에서 B를 가리키는 화살표가 모두 사라졌어 (B 자신은 여전히 A, C를 가리키지만 아무도 B를 안 봐)
- ⑥ **`current.prev`(A)**

**단계 4가 필요한 이유**
- ⑦ **더미** (머리의 앞쪽은 더미니까)
- ⑧ **`None`** (더미의 data)

**실측**
```
머리 A 삭제 → current = B
중간 B 삭제 → current = A
꼬리 C 삭제 → current = B
```
- ⑨ **B** ⑩ **A** ⑪ **B**

**단계 4를 빼면**: 머리를 삭제했을 때 `current` 가 더미가 되고, `print_current_node()` 가 **`None` 을 출력**해. 실측에서 확인했지.

> 🔑 29일차 17번 원리 **"마지막으로 손댄, 리스트에 남아 있는 노드"** 의 오늘 버전:
> **"삭제한 노드의 앞쪽. 단 그게 더미면 머리로 보정."** 더미는 데이터 노드가 아니니까.

---

### 12. 2줄짜리 함수 4개

| 함수 | current 를 어디로 | 왜 |
|---|---|---|
| `add_first` | ① `head`(더미) | ② **`add` 는 "바로 뒤"에 넣으니, 더미 뒤 = 맨 앞** |
| `add_last` | ③ **`head.prev`(꼬리)** | ④ **꼬리 뒤 = 맨 끝** |
| `remove_first` | ⑤ **`head.next`(머리)** | ⑥ **`remove_current_node` 는 자신을 지우니, 머리를 지정** |
| `remove_last` | ⑦ **`head.prev`(꼬리)** | ⑧ **꼬리를 지정** |

- ⑨ **`add` 는 "current 바로 뒤"에 삽입하고, `remove_current_node` 는 "current 자신"을 삭제하기 때문.** 기준점이 다르니 지정하는 위치도 달라져.
- ⑩ **꼬리 뒤에 넣는 것 = 맨 끝 삽입**, **꼬리 자신을 지우는 것 = 맨 끝 삭제**. 우연히 같은 노드를 가리키게 된 거야.

---

### 13. 🔴 `remove(p)` 가 O(n)인 이유
- ① **O(1)**
- ② **`p` 가 이 리스트에 실제로 존재하는지 검증하기 위해서.** 다른 리스트의 노드나 이미 삭제된 노드를 넘겨받으면, 직접 방식은 **`no` 만 줄이고 엉뚱한 링크를 건드려** 자료구조가 깨져.

| | 교재 (탐색) | 빠른 (직접) |
|---|---|---|
| 시간 복잡도 | ③ **O(n)** | ④ **O(1)** |
| `p` 가 없는 노드일 때 | ⑤ **아무 일도 안 함 (안전)** | ⑥ **`no` 가 잘못 줄어듦** |
| 안전성 | ⑦ **높음** | ⑧ **호출자를 신뢰해야 함** |

**실측**: 리스트에 없는 노드 `Z` 를 지우면 교재 방식은 `no=3` 유지, 빠른 방식은 `no=2` 로 깨져.

- **29일차 11번과의 차이**: 그때는 `prev` 가 없어서 **어쩔 수 없이** 훑었어. 오늘은 **선택**이야. 안전을 살지, 속도를 살지.
- 실무에서는 보통 O(1) 버전을 쓰되, **노드 핸들을 외부에 노출하지 않는 방식**으로 안전을 확보해.

---

## 💻 PART 4 해설

### 14~16, 18. 빈칸 정답

**14번**
```python
self.data = data                      # ①
self.prev = prev or self              # ②
self.next = next or self              # ③
self.head = self.current = MyNode()   # ④
return self.no                        # ⑤
return self.head.next is self.head    # ⑥
```

**15번**
```python
ptr = self.head.next                  # ①  ← head가 아니다!
while ptr is not self.head:           # ②  ← None이 아니다!
    ptr = ptr.next                    # ③
return -1                             # ④
self.current.next is self.head        # ⑤  뒤쪽이 더미 = 꼬리
self.current = self.current.next      # ⑥
self.current.prev is self.head        # ⑦  앞쪽이 더미 = 머리
self.current = self.current.prev      # ⑧
```

**16번**
```python
node = MyNode(data, self.current, self.current.next)   # ①②
self.current.next.prev = node         # ③  뒤쪽부터! (17번)
self.current.next = node              # ④
self.current = self.head              # ⑤
self.current = self.head.prev         # ⑥  O(1)!
```

**18번**
```python
self.current.prev.next = self.current.next   # ①
self.current.next.prev = self.current.prev   # ②
self.current = self.current.prev             # ③
if self.current is self.head:                # ④
    self.current = self.head.next            # ⑤
self.current = self.head.next                # ⑥
self.current = self.head.prev                # ⑦
self.current = head.next                     # ⑧
if self.current is self.head:                # ⑨
self.current = self.current.next             # ⑩
self.current = head.prev                     # ⑪
self.current = self.current.prev             # ⑫
```

---

### 17. 🔥🔥 보충수업 8-4 — 연속 대입 함정

**실측 결과가 충격적이야**
```
교재 (2줄):
  정방향: 더미 ⇄ [A] ⇄ [B] ⇄ [C] ⇄ 더미
  역방향: 더미 ⇄ [C] ⇄ [B] ⇄ [A] ⇄ 더미     ✅

한 줄 연속대입:
  정방향: 더미 ⇄ [C] ⇄ [B] ⇄ [A] ⇄ 더미     ← 순서가 뒤집혔다!
  역방향: (아무것도 없음)                      ← 역방향이 통째로 죽었다 💥
```

- ① **순서가 거꾸로 (C → B → A)**
- ② **역방향 링크가 전부 끊어져서 빈 것처럼 보임**
- ③ **에러 없이 조용히 깨진다** 🔥

**왜 이렇게 되나**
```python
self.current.next = self.current.next.prev = node
```
파이썬은 `a = b = expr` 을 이렇게 처리해:
1. `expr`(= `node`)를 **한 번만** 평가
2. **왼쪽부터** 차례로 대입 → `self.current.next = node` **먼저**
3. 그다음 `self.current.next.prev = node` — 그런데 `self.current.next` 는 **이미 `node`**
4. 결국 **`node.prev = node`** 가 되어 자기 자신을 가리킴

**최소 재현**
```
한 줄:  a.link=new, new.link=new, b.link=None    ❌
두 줄:  a2.link=new, new2.link=None, b2.link=new  ✅
```

> 🔑 **C/Java는 `=` 가 오른쪽 결합**이라 뒤에서부터 대입되어 잘 동작해. 파이썬은 **왼쪽부터**라 반대야.
> 교재 371p: **"파이썬에서 '='은 연산자가 아닙니다. 그러므로 당연히 오른쪽 결합 연산자도 아닙니다."**

**26일차 2번과의 비교** 💡
```python
pl, pr = left, right = range.pop()     # 26일차 — 문제없음
```
같은 문법인데 왜 그때는 괜찮았을까? **대입 대상이 단순 변수**라서 서로 영향을 주지 않았거든. 오늘은 **대입 대상 자체가 다른 대입의 결과에 의존**해서 터진 거야.

---

## 📐 PART 5 해설

### 19. 🔥 `add_last` 가 O(1)

**실측**
```
      n | 08-2 (29일차) | 08-4 (오늘) |  배수
    500 |       2.09ms |     0.44ms |   4.8배
   1000 |       7.69ms |     1.00ms |   7.7배
   2000 |      31.19ms |     1.93ms |  16.1배
   4000 |     125.54ms |     4.54ms |  27.6배
```

- ① 08-2: n이 2배면 시간은 **약 4배** → n개 삽입이 **O(n²)**
- ② 08-4: n이 2배면 시간도 **약 2배** → n개 삽입이 **O(n)**, 즉 1회가 **O(1)**
- **배수가 계속 커지는 게 증거**야. 4.8 → 7.7 → 16.1 → 27.6. 두 알고리즘의 복잡도 차수가 다르다는 뜻이지.

**이유**: `head.prev` 가 곧 꼬리라서 `while` 탐색이 사라졌어. 29일차 21번에서 `tail` 포인터를 추가하면 O(1)이 된다고 했는데, **원형 구조가 그걸 공짜로 제공**하는 거야.

---

### 20. `clear()` 의 `self.no = 0`
- ① **0** ② **불필요 (안전장치)**

**세 버전 비교**
- 08-2 (29일차): `no = 0` **있음** — 근데 `remove_first` 버그 때문에 **필요했음**
- 08-3 (30일차): `no = 0` **없음** — `remove_first` 가 정상이라 불필요
- 08-4 (오늘): `no = 0` **있음** — 정상이지만 남겨둠

**실측**: `remove_first` 반복만으로 `no` 가 정확히 0이 돼. 그리고 빈 리스트에서 삭제 3종을 9번 호출해도 `no = 0` 유지 — **29일차 20번 버그가 오늘은 없어.**

---

### 21. 08장 완전 총정리

| | 08-2 포인터 | 08-3 커서 | 08-4 원형이중 |
|---|---|---|---|
| 노드의 정체 | 객체 | 배열 원소 | ① **객체** |
| "없음" 표시 | `None` | `Null(-1)` | ② **더미 노드** |
| `add_first` | O(1) | O(1) | ③ **O(1)** |
| `add_last` | O(n) | O(n) | ④ **O(1)** 🔥 |
| `remove_last` | O(n) | O(n) | ⑤ **O(1)** 🔥 |
| 앞쪽 이동 | ❌ | ❌ | ⑥ **✅** |
| 역순 순회 | ❌ | ❌ | ⑦ **✅** |
| 크기 제한 | 없음 | ⑧ **capacity** | ⑨ **없음** |
| 빈 리스트 분기 | 많음 | 많음 | ⑩ **없음** |
| 노드당 메모리 | data+next | data+next+dnext | ⑪ **data+prev+next** |

**최종 질문 답**

**1. 모든 면에서 나은가?** 거의 그래. 대가는 두 가지:
- **노드당 포인터가 하나 더** (`prev`) → 메모리 약 1.5배
- **코드가 더 정교해야 함** — 양방향 링크를 항상 쌍으로 갱신해야 하고, 17번 같은 함정이 있어

**2. 더미가 없앤 두 가지**
- **빈 리스트 분기** (`if head is None`)
- **`head` 갱신** (머리가 바뀌어도 `head` 는 그대로)

**3. `deque` 가 O(1)인 이유**
양쪽 끝에 **O(1)로 접근할 수 있는 구조**(양방향 링크)를 갖고 있어서, `appendleft`/`popleft`/`append`/`pop` 모두 탐색 없이 링크만 바꾸면 돼. 오늘 우리가 만든 것과 발상이 같아. (실제 CPython은 성능을 위해 **블록 단위 이중 연결 리스트**를 쓰지만 원리는 동일)

**실측**: 네 연산 모두 우리 구현 대비 `deque` 가 수십 배 빠른데, 이건 **C 구현이라서**지 알고리즘이 달라서가 아니야.

**4. 비유 바꾸기**
29일차: "A가 B에게, B가 C에게 차례로 연락하는 비상 연락망. 뒤돌아 앞 사람에게 연락할 수 없다."
→ 오늘: **"둥글게 앉아 양옆 사람을 모두 아는 원탁."** 누구든 좌우로 갈 수 있고, 한 바퀴 돌면 제자리로 와. 그리고 **사회자(더미)가 한 명 앉아 있어서** "첫 번째"와 "마지막"이 특별하지 않아.

---

## 📌 핵심 3줄 요약

1. **원형 + 이중 + 더미, 세 아이디어가 29~30일차의 약점 세 개를 정확히 하나씩 없앤다.** 원형이라 `head.prev` 가 곧 꼬리 → `add_last` 가 O(n)에서 **O(1)** (실측 n=4000에서 27.6배), 이중이라 `prev` 로 앞쪽 이동과 역순 순회 가능, 더미라 빈 리스트 분기와 `head` 갱신이 **전부 사라짐**.
2. **더미 노드의 본질은 "경계를 특별 취급하지 않게 만드는 것"이다.** `current.next` 가 항상 존재하므로 `None` 검사가 0개가 되고, `add_first`/`add_last`/`remove_first`/`remove_last` 가 전부 **2줄**로 줄어들어.
3. **파이썬의 `a = b = expr` 은 왼쪽부터 대입한다.** C/Java와 반대라서 `self.current.next = self.current.next.prev = node` 를 한 줄로 쓰면 `node.prev = node` 가 되어 **역방향 링크가 통째로 죽는다.** 그런데 에러는 안 나.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 5, 6, 9, 12, 14, 20, 21번)**: 전원 필수
  - **5~6번 더미 노드**를 잡아야 나머지가 다 읽혀
  - **21번 총정리표**는 08장 전체를 한 장으로 압축한 것
- 🟡 **(R-2, 3, 4, 7, 10, 11, 15, 16, 19번)**: 팀 목표선
  - **7번(더미가 없애는 분기)** 이 오늘의 개념 고비 🔥
  - **4번 참조식 표**는 원형+이중의 감을 잡는 최고의 연습
  - **19번 실측**은 꼭 돌려볼 것 — 배수가 4.8→27.6으로 커지는 게 O(n²) vs O(n)의 증거
- 🔴 **(8, 13, 17, 18번)**: 도전
  - **17번이 오늘 최대 수확** 🔥🔥 — 역방향이 통째로 죽는 걸 직접 볼 것. 파이썬 개발자면 평생 한 번은 만나는 함정
  - **13번(`remove` 의 O(n) vs O(1))** 은 "안전 vs 속도" 트레이드오프를 판단하는 훈련
  - **8번**은 더미의 가치를 반대편에서 확인하는 문제
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 21번)

## 🔗 오늘 회수된 개념들

- **29일차 3번 "앞으로 못 간다"** → `prev` 포인터로 해결 (2번)
- **29일차 10번 `pre` 두 커서** → `p.prev` 하나로 대체 (2, 13번)
- **29일차 21번 `add_last` O(n²)** → `head.prev` 로 O(1) (19번)
- **29일차 17번 current 원리** → "삭제한 노드의 앞쪽, 더미면 머리로 보정" (11번)
- **29일차 20번 `no` 음수 버그** → 오늘은 `if not is_empty()` 로 완전 차단 (20번)
- **29일차 18~19번 이터레이터** → 오늘은 역순까지 (9번)
- **30일차 19번 `or` 연산자** → `prev or self` 로 원형의 씨앗 (R-2)
- **30일차 8번 프리 리스트** → 오늘은 배열이 아니라 객체라 불필요
- **26일차 2번 연쇄 대입** → 오늘은 그게 함정이 되는 경우 (17번)
- **3일차 `is` vs `==`** → 노드 비교는 `is` (13번)

---

> 🎉 **08장 리스트 완주!** 배열 리스트의 한계 → 포인터 → 커서 → 원형 이중까지, **네 가지 구현**을 전부 손으로 짰어.
>
> **이번 3일의 흐름**: "객체로 잇기 → 배열 안에서 잇기 → 양방향으로 둥글게 잇기"
> 같은 문제(순서 있는 데이터 관리)를 푸는 세 가지 설계였고, **뒤로 갈수록 분기가 사라지고 코드가 짧아졌지.**
>
> **다음 진도**: 09장 **트리** — 오늘까지는 노드가 **한 줄로** 늘어섰지만, 트리는 **갈라져.** 24일차 힙 정렬에서 맛본 그 구조가 본격적으로 나와.